In [1]:
library(tidyverse)
library(repr)
library(paletteer)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.2
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
tl_raw<- read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Tissue_Loss/T1-T4_Avg_Tissue_Loss.csv',show_col_types = FALSE)

In [3]:
tl_raw<-tl_raw %>%
    select(1:14) %>%
    select(-`immune_y/n`,-Height,-MaxDiameter,-Size_Class)
# create colony_id
tl_raw<-tl_raw %>%
    mutate(colony_id = paste0("T",TransectNum,"_",Species,'_',NewTagNum))

In [4]:
# combine with colony data for health statuses
colony<-read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Sample_Data/CBC_ColonyData.csv',show_col_types = FALSE)

In [5]:
colony<-colony %>%
    select(TransectNum,NewTagNum,Species,`062019_Condition`,`102019_Condition`,`052022_Condition`,`122022_Condition`) %>%
    filter(!(TransectNum %in% c('5','6')))
# create colony_id
colony<-colony %>%
    mutate(colony_id = paste0("T",TransectNum,"_",Species,'_',NewTagNum))

In [6]:
head(colony,2)

TransectNum,NewTagNum,Species,062019_Condition,102019_Condition,052022_Condition,122022_Condition,colony_id
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,SSID,Healthy,NA,Diseased,Diseased,T1_SSID_1
1,2,PAST,Healthy,NA,Healthy,Healthy,T1_PAST_2


In [7]:
# merge by colony id 
tl<-tl_raw %>% 
    left_join(colony %>% select(-TransectNum,-NewTagNum,-Species), by = "colony_id")

In [8]:
# select diseased colonies only
conditions <- c('Diseased','Dead')
tl_dis<-tl %>%
    filter(`052022_Condition` %in% conditions | `122022_Condition` %in% conditions)

In [9]:
# pivot
tl_dis_long <- tl_dis %>% 
  pivot_longer(
    cols = matches("^\\d{6}_(Condition|TL)$"),
    names_to = c("Date", ".value"),
    names_pattern = "^(\\d{6})_(Condition|TL)$"
  )

In [10]:
head(tl_dis_long,6)

Date_InitialTag,Transect,TransectNum,NewTagNum,Species,Date_DocumentedDisease,Date_DocumentedMortality,colony_id,Date,TL,Condition
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,062019,82.5,Healthy
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,052022,37.5,Diseased
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,122022,25,Diseased
6/21/2019,CBC30N,1,1,SSID,5/21/2022,Diseased,T1_SSID_1,102019,NA,NA
6/21/2019,CBC30N,1,3,SSID,5/21/2022,Diseased,T1_SSID_3,062019,95,Healthy
6/21/2019,CBC30N,1,3,SSID,5/21/2022,Diseased,T1_SSID_3,052022,95,Diseased


In [11]:
# calculate rate of tl for diseased colonies
# change in TL over time
    # collect exact monitoring date
    # calculate number of weeks in between time points
    # calculate change in TL between time points
    # May and Dec: change in TL / number of weeks

In [12]:
# collect exact monitoring date

In [13]:
# add sampling date
samples<-read_csv('/Users/brookesienkiewicz/Documents/sctld/SCTLD_samples/Sample_Data/CBC_samples.csv',show_col_types = FALSE)

In [14]:
samples<-samples %>% 
    select(Month_year,CollectionDate,TransectNum,NewTagNum,Species) %>%
    distinct() %>%
    rename(Date = Month_year) %>%
    mutate(colony_id = paste0("T",TransectNum,"_",Species,'_',NewTagNum))

In [15]:
tld<-tl_dis_long %>% 
    filter(!(is.na(Condition))) %>%
    left_join(samples %>% select(-TransectNum,-NewTagNum,-Species), by = c("colony_id","Date"))

In [16]:
# add in monitoring date
tld<-tld %>%
    mutate(fulldate = as.Date(CollectionDate,format = "%m/%d/%y"))
# if missing collectiondate,
# grab another fulldate from coral with same timepoint
tld <- tld %>%
  group_by(Date, Transect) %>%
  fill(fulldate, .direction = "downup") %>%
  ungroup()

In [17]:
unique(tld$TL)

[1] "82.5"    "37.5"    "25"      "95"      "92.5"    "97.5"    "35"     
 [8] "0"       NA        "27.5"    "100"     "62.5"    "15"      "7.5"    
[15] "5"       "2.5"     "70"      "22.5"    "90"      "65"      "77.5"   
[22] "50"      "52.5"    "exclude" "60"      "55"      "32.5"    "67.5"   
[29] "n/a"     "45"      "20"      "10"      "85"      "40"      "57.5"   
[36] "87.5"

In [18]:
tld$TL <- as.numeric(tld$TL)
# remove NAs (entries were either NA or 'exclude' already)
tld <- tld %>% filter(!(is.na(TL)))

Warning message:
“NAs introduced by coercion”


In [19]:
# drop colonies after they die
# id colonies with 0 tissue twice in a row and remove second time point
tld<- tld %>%
  group_by(colony_id) %>%
  arrange(fulldate, .by_group = TRUE) %>%
  mutate(dead_flag = cummax(as.numeric(TL == 0 & lag(TL, default = first(TL)) == 0))) %>%
  filter(dead_flag == 0) %>%
  select(-dead_flag) %>%
  ungroup()

In [20]:
# calculate number of weeks in between time points
tld <- tld %>%
    group_by(colony_id) %>%
    arrange(fulldate, .by_group = TRUE) %>%
    mutate(diff_weeks = as.numeric(difftime(fulldate, lag(fulldate), units = "weeks"))) %>%
# calculate change in TL between time points
    mutate(diff_tl = TL - lag(TL)) %>%
    ungroup()

In [21]:
# calculate TL rates
tld<-tld %>%
    mutate(rate = round(diff_tl / diff_weeks,2))

In [ ]:
# avg TL rates per spp (2019 -> May, May -> Dec)
ratei<-tld %>%
    filter(Date == "052022") %>%
    group_by(Species) %>%
    summarise(mean_ratei = round(mean(rate, na.rm = TRUE),2)) %>%
    arrange(mean_ratei)

In [ ]:
# avg TL rates per spp (May -> Dec)
ratef<-tld %>%
    filter(Date == "122022") %>%
    group_by(Species) %>%
    summarise(mean_ratef = round(mean(rate, na.rm = TRUE),2)) %>%
    arrange(mean_ratef)

In [31]:
left_join(ratei,ratef, by = "Species")

Species,mean_ratei,mean_ratef
<chr>,<dbl>,<dbl>
MMEA,-0.65,NA
MCAV,-0.49,-1.08
PSTR,-0.48,-1.63
SSID,-0.31,-0.33
PAST,-0.27,-0.32
DLAB,NaN,-0.18
OANN,NaN,-0.09
OFAV,NaN,-0.36
